In [2]:
  # Cell 1: Import Libraries
import cv2
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import joblib

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
   # Cell 2: Feature Extraction Function
def extract_features(image_path):
    """Extract features from a rice grain image"""
    img = cv2.imread(image_path)
    if img is None:
        return None
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    
    # Thresholding to separate grain from background
    _, thresh = cv2.threshold(blurred, 100, 255, cv2.THRESH_BINARY_INV)
    
    # Morphological operations
    kernel = np.ones((3,3), np.uint8)
    opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)
    
    # Find contours
    contours, _ = cv2.findContours(opening, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours:
        return None
    
    # Get largest contour (main grain)
    c = max(contours, key=cv2.contourArea)
    
    # Extract features
    area = cv2.contourArea(c)
    perimeter = cv2.arcLength(c, True)
    x, y, w, h = cv2.boundingRect(c)
    aspect_ratio = float(w)/h if h > 0 else 0
    
    # Color features
    mask = np.zeros(gray.shape, np.uint8)
    cv2.drawContours(mask, [c], -1, 255, -1)
    mean_color = cv2.mean(img, mask=mask)[:3]  # B, G, R
    
    return [area, perimeter, w, h, aspect_ratio, mean_color[0], mean_color[1], mean_color[2]]

print("Feature extraction function created!")

Feature extraction function created!


In [4]:
# Cell 3: Load Dataset and Extract Features
# IMPORTANT: You need to have a dataset folder with images
# Folder structure:
# dataset/
#   ├── whole/
#   ├── broken/
#   └── defective/

dataset_path = 'dataset'
categories = ['whole', 'broken', 'defective']

data = []
labels = []

for category in categories:
    category_path = os.path.join(dataset_path, category)
    if not os.path.exists(category_path):
        print(f"Warning: {category_path} not found. Creating empty folder.")
        os.makedirs(category_path, exist_ok=True)
        continue
    
    for img_name in os.listdir(category_path):
        if img_name.endswith(('.jpg', '.jpeg', '.png')):
            img_path = os.path.join(category_path, img_name)
            features = extract_features(img_path)
            if features is not None:
                data.append(features)
                labels.append(category)
                print(f"Processed: {img_name} ({category})")

if len(data) > 0:
    # Create DataFrame
    df = pd.DataFrame(data, columns=['area', 'perimeter', 'width', 'height', 
                                      'aspect_ratio', 'B', 'G', 'R'])
    df['label'] = labels
    
    print(f"\nTotal samples: {len(df)}")
    print(f"Samples per category:")
    print(df['label'].value_counts())
    print(f"\nFirst few rows:")
    print(df.head())
else:
    print("No images found! Please add images to the dataset folders.")

Processed: entero_grano1.png (whole)
Processed: entero_grano10.png (whole)
Processed: entero_grano100.png (whole)
Processed: entero_grano1000.png (whole)
Processed: entero_grano1001.png (whole)
Processed: entero_grano1002.png (whole)
Processed: entero_grano1003.png (whole)
Processed: entero_grano1004.png (whole)
Processed: entero_grano1005.png (whole)
Processed: entero_grano1006.png (whole)
Processed: entero_grano1007.png (whole)
Processed: entero_grano1008.png (whole)
Processed: entero_grano1009.png (whole)
Processed: entero_grano101.png (whole)
Processed: entero_grano1010.png (whole)
Processed: entero_grano1011.png (whole)
Processed: entero_grano1012.png (whole)
Processed: entero_grano1013.png (whole)
Processed: entero_grano1014.png (whole)
Processed: entero_grano1015.png (whole)
Processed: entero_grano1016.png (whole)
Processed: entero_grano1017.png (whole)
Processed: entero_grano1018.png (whole)
Processed: entero_grano1019.png (whole)
Processed: entero_grano102.png (whole)
Processe

In [5]:
# Cell 4: Train and Save Model
if len(data) > 0:
    X = df[['area', 'perimeter', 'width', 'height', 'aspect_ratio', 'B', 'G', 'R']]
    y = df['label']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
    
    # Save the model
    joblib.dump(model, 'rice_model.pkl')
    print("\n✅✅✅ Model saved as 'rice_model.pkl' ✅✅✅")
else:
    print("❌ Cannot train model - no data available!")

Classification Report:
              precision    recall  f1-score   support

      broken       0.79      0.77      0.78       176
   defective       0.89      0.84      0.86       177
       whole       0.93      0.95      0.94       662

    accuracy                           0.90      1015
   macro avg       0.87      0.85      0.86      1015
weighted avg       0.90      0.90      0.90      1015

Accuracy: 89.85%

✅✅✅ Model saved as 'rice_model.pkl' ✅✅✅
